In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("emails.csv")
data.head()

,Email_ID,Subject,Message,Label
0,1,Win a free iPhone!,Congratulations! You have won an iPhone. Click...,Spam
1,2,Meeting Reminder,"Dear team, the meeting is scheduled for tomorrow.",Non-Spam
2,3,Claim Your Lottery Prize,"You won $1,000,000! Provide details to claim now.",Spam
3,4,Invoice Attached,Your invoice for January is attached. Check de...,Non-Spam
4,5,URGENT: Account Suspended,Your bank account is suspended. Verify your de...,Spam


In [3]:
data.Label.value_counts()

Label
Spam        16
Non-Spam    14
Name: count, dtype: int64

In [4]:
def get_spam_email(x):
    if x == 'Spam':
        return 1
    return 0


In [5]:
data['Spam_Flag'] = data['Label'].apply(get_spam_email)

In [6]:
data.head()

,Email_ID,Subject,Message,Label,Spam_Flag
0,1,Win a free iPhone!,Congratulations! You have won an iPhone. Click...,Spam,1
1,2,Meeting Reminder,"Dear team, the meeting is scheduled for tomorrow.",Non-Spam,0
2,3,Claim Your Lottery Prize,"You won $1,000,000! Provide details to claim now.",Spam,1
3,4,Invoice Attached,Your invoice for January is attached. Check de...,Non-Spam,0
4,5,URGENT: Account Suspended,Your bank account is suspended. Verify your de...,Spam,1


In [7]:
from sklearn.model_selection import train_test_split

X = data.Message
Y= data.Spam_Flag
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.2, random_state= 42)

In [8]:
X_train[:4]

28    Your account needs verification. Click to verify.
24    You have been selected for a special rewards p...
12            Claim your prize now! Only 24 hours left.
0     Congratulations! You have won an iPhone. Click...
Name: Message, dtype: object

In [9]:
Y_train[:4]

28    1
24    1
12    1
0     1
Name: Spam_Flag, dtype: int64

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer()

X_train_cv = v.fit_transform(X_train.values)
X_train_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 186 stored elements and shape (24, 131)>

In [11]:
print("X_train =", X_train_cv.toarray()[:2][0])

X_train = [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1]


In [12]:
v.vocabulary_

{'your': 130,
 'account': 7,
 'needs': 72,
 'verification': 122,
 'click': 24,
 'to': 111,
 'verify': 123,
 'you': 129,
 'have': 44,
 'been': 17,
 'selected': 98,
 'for': 39,
 'special': 103,
 'rewards': 94,
 'program': 90,
 'claim': 23,
 'prize': 88,
 'now': 76,
 'only': 80,
 '24': 4,
 'hours': 48,
 'left': 61,
 'congratulations': 26,
 'won': 128,
 'an': 10,
 'iphone': 54,
 'here': 45,
 'bank': 16,
 'is': 55,
 'suspended': 105,
 'details': 29,
 'free': 40,
 'vacation': 121,
 'in': 50,
 'the': 108,
 'maldives': 68,
 'sign': 101,
 'up': 118,
 'hey': 46,
 'are': 13,
 'we': 125,
 'still': 104,
 'on': 79,
 'dinner': 31,
 'tonight': 114,
 'order': 82,
 'has': 43,
 'shipped': 100,
 'track': 115,
 'it': 56,
 'weekly': 127,
 'newsletter': 74,
 'from': 41,
 'techworld': 107,
 'get': 42,
 '50': 5,
 'off': 77,
 'holiday': 47,
 'bookings': 19,
 'limited': 63,
 'period': 86,
 'offer': 78,
 'dear': 28,
 'team': 106,
 'meeting': 70,
 'scheduled': 96,
 'tomorrow': 113,
 '000': 0,
 'provide': 91,
 'our

In [13]:
from sklearn.naive_bayes import MultinomialNB

model= MultinomialNB()
model.fit(X_train_cv, Y_train)

MultinomialNB()

In [14]:
X_test_cv = v.transform(X_test)
y_pred = model.predict(X_test_cv)


In [15]:
from sklearn.metrics import classification_report
print("Classification report \n", classification_report(Y_test, y_pred))

Classification report 
               precision    recall  f1-score   support

           0       1.00      0.80      0.89         5
           1       0.50      1.00      0.67         1

    accuracy                           0.83         6
   macro avg       0.75      0.90      0.78         6
weighted avg       0.92      0.83      0.85         6



In [16]:
emails = ["Congratulations! You've Won a $500 Gift Card!",
          "Meeting Reminder for Tomorrow",
          "URGENT: Your Bank Account is Suspended!"]
email_count = v.transform(emails)
model.predict(email_count)

array([1, 0, 1])

In [23]:
# using sklearn.pipeline
from sklearn.pipeline import Pipeline
clf = Pipeline([
    ('vectorizer', CountVectorizer()), 
    ('nb', MultinomialNB())
])

clf

Pipeline(steps=[('vectorizer', CountVectorizer()), ('nb', MultinomialNB())])

In [24]:
clf.fit(X_train, Y_train)

Pipeline(steps=[('vectorizer', CountVectorizer()), ('nb', MultinomialNB())])

In [26]:
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

    non-spam       0.00      0.00      0.00       2.0
        spam       0.00      0.00      0.00       0.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_pr

In [21]:
# trial

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Sample data (text and labels)
X = [
    "Win a free iPhone!", 
    "Meeting reminder for tomorrow", 
    "Claim your $500 prize now!", 
    "Project meeting at 10 AM tomorrow"
]
Y = ['spam', 'non-spam', 'spam', 'non-spam']

# Split data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Create pipeline
clf = Pipeline([
    ('vectorizer', CountVectorizer()), 
    ('nb', MultinomialNB())
])

# Fit the model
clf.fit(X_train, Y_train)

# Now you can make predictions
predictions = clf.predict(X_test)
print(predictions)


['spam' 'spam']
